# Shortest and Widest Paths

This example computes all-pairs shortest paths and widest paths on a small directed graph, and then answers
single-source and single-target queries from one factorization.

In [ ]:
using PATHPACK, SparseArrays, LinearAlgebra
Base.include(@__MODULE__, joinpath(pkgdir(PATHPACK), "docs", "literate", "_figs.jl"))  #hide
using PATHPACK.CPU: MinPlus, MaxMin, mstar, mlu

The entry `A[i, j]` is the weight of the arc `i → j`.

In [ ]:
A = sparse([1, 1, 2, 3, 4, 4], [2, 3, 4, 2, 1, 5], [3.0, 8.0, 1.0, 4.0, 2.0, 7.0], 5, 5)

## All Pairs

In the min-plus semiring the star `A*` holds the distances: `A*[i, j]` is the length of a shortest path from `i` to
`j`, and `Inf` means that `j` cannot be reached from `i`.

In [ ]:
D = mstar(MinPlus(), A)

In the max-min semiring the same call gives the capacity of the widest path between every pair of vertices.

In [ ]:
W = mstar(MaxMin(), A)

In [ ]:
figure(panel("mstar(MinPlus(), A)", matrixfig(D; val = Figs.unit(D)); sub = "shortest distance"),  #hide
    panel("mstar(MaxMin(), A)", matrixfig(W; val = 1 .- Figs.unit(W)); sub = "widest capacity");  #hide
    caption = "Row i, column j: from vertex i to vertex j. Inf: no path; in max-min the diagonal is the semiring's one, Inf.")  #hide

## Queries

A factorization answers queries without forming the whole star. `lmul!(F, b)` overwrites `b` with `A* b`; with the
unit vector at vertex 5 (`0` there, the semiring's one, and `Inf`, its zero, elsewhere), that is column 5 of `A*`:
the distances to vertex 5.

In [ ]:
F = mlu(MinPlus(), A)
b = fill(Inf, 5); b[5] = 0.0
lmul!(F, b)

`rmul!(B, F)` overwrites the rows of `B` with `B A*`; with the unit row vector at vertex 1, that is row 1 of `A*`:
the distances from vertex 1.

In [ ]:
B = fill(Inf, 1, 5); B[1] = 0.0
rmul!(B, F)

Both agree with the star.

In [ ]:
b == D[:, 5] && vec(B) == D[1, :]